# Lesson B — Long-Term Memory (Cross-Session)
**Presented by:** Sharad Rajore | **Organization:** Zensar Technologies

---

### 🎯 Learning Objectives
1. **See the Problem:** Even with disk-persisted checkpoints (Lesson A), a *new* `thread_id` for the same returning user starts with zero knowledge of them.
2. **Approach 1 — `InMemoryStore`:** Store facts keyed by `user_id`, independent of any single conversation thread.
3. **Approach 2 — Memory Tools:** Let the agent decide *what* to remember and *when* to recall it, using `save_memory` / `recall_memories` tools.
4. **Approach 3 — Semantic Recall:** Retrieve relevant memories by meaning, not exact keys, using embeddings.
5. **Approach 4 — `SqliteStore`:** Persist long-term memory to disk for production.

### 🧠 The Core Idea
Lesson A solved *short-term* memory: replaying a conversation within one `thread_id`.
That's still **session-scoped** — a new thread (a returning customer calling back next week, a new chat tab) starts blank.

**Long-term memory** is a *separate* store, keyed by **who the user is** (`user_id`), not *which conversation* they're in (`thread_id`). It holds distilled facts — not the whole transcript.

| Aspect | Short-term (Lesson A) | Long-term (this lesson) |
|---|---|---|
| Mechanism | `checkpointer` | `store` |
| Keyed by | `thread_id` | `user_id` (namespace) |
| What's stored | Full message history | Extracted facts / preferences |
| Survives a new thread? | ❌ No | ✅ Yes |
| Retrieval | Replays everything | Searches for what's relevant |


## ⚙️ Setup

In [1]:
from dotenv import load_dotenv
load_dotenv()

import langchain
print(f"LangChain: {langchain.__version__}")


LangChain: 1.3.9


In [2]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="gpt-oss:120b-cloud", temperature=0)

# -- Alternatively, use Groq --------------------------------------------------
# from langchain_groq import ChatGroq
# llm = ChatGroq(model="llama-3.3-70b-versatile")
# ------------------------------------------------------------------------------

print("LLM ready.")


LLM ready.


In [3]:
from langchain_core.tools import tool

# Reusing the familiar tools from previous lessons
@tool
def get_weather(city: str) -> str:
    """Returns the current weather for a given city."""
    return f"The weather in {city} is sunny with a high of 28C."

@tool
def get_stock_price(ticker: str) -> str:
    """Returns the current stock price for a given ticker symbol.
    Use 'ZENSAR' for Zensar Technologies, 'GOOGL' for Google."""
    prices = {"ZENSAR": "464.00 INR", "GOOGL": "175.00 USD"}
    return prices.get(ticker.upper(), f"Unknown ticker: {ticker}")

print("Tools ready: get_weather, get_stock_price")


Tools ready: get_weather, get_stock_price


---
## Part 1: The Problem — A Stranger Every New Session

Let's reuse the `SqliteSaver`/`MemorySaver` pattern from Lesson A — disk-persisted, thread-scoped memory.
Then we simulate a **returning user opening a brand-new conversation** (a new `thread_id`), exactly like a customer starting a fresh chat session next week.

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

checkpointer_only = MemorySaver()

agent_thread_only = create_agent(
    model=llm,
    tools=[get_weather, get_stock_price],
    system_prompt="You are a helpful assistant.",
    checkpointer=checkpointer_only,
)

# Session 1 -- thread "chat_2026_07_01"
cfg_session1 = {"configurable": {"thread_id": "chat_2026_07_01"}}
r1 = agent_thread_only.invoke(
    {"messages": [("user", "Hi, I'm Sharad. I work at Zensar Technologies and I prefer Celsius for weather.")]},
    config=cfg_session1,
)
print("Session 1 -- Agent:", r1["messages"][-1].content)

# Session 2 -- SAME PERSON, but a NEW thread_id (e.g. they opened a new chat tab)
cfg_session2 = {"configurable": {"thread_id": "chat_2026_07_08"}}
r2 = agent_thread_only.invoke(
    {"messages": [("user", "What do you know about me?")]},
    config=cfg_session2,
)
print("Session 2 (new thread) -- Agent:", r2["messages"][-1].content)
print()
print("A new thread_id = a total stranger, even though it's the same returning user.")


Session 1 -- Agent: Nice to meet you, Sharad! 👋 I’ve noted that you work at Zensar Technologies and that you prefer temperatures in Celsius. How can I help you today—perhaps a weather update for a specific city, a stock look‑up for Zensar, or something else? Let me know!
Session 2 (new thread) -- Agent: I don’t have any information about you beyond what you choose to share during our conversation. If there’s something specific you’d like me to know—or if you have a question about a particular topic—I’m happy to help!

A new thread_id = a total stranger, even though it's the same returning user.


### 🔍 Why does this happen?

`checkpointer` state is **namespaced by `thread_id`**. That's exactly what makes multi-user isolation work (Lesson A) — but it also means *every new thread starts empty*, even for a user the agent has spoken to a hundred times before.

To fix this we need a **second, independent store** — one that's keyed by **who the user is**, and that the agent can read from *before* it even knows which thread it's in.

---
## Approach 1 — `InMemoryStore`: Facts Keyed by User

LangGraph ships a `store` primitive, separate from the `checkpointer`. Think of it as a small key-value + search database, organized into **namespaces** — typically `(memory_type, user_id)`.

In [5]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

# Namespace = a tuple, commonly ("memories", user_id)
namespace = ("memories", "user_sharad")

# put(namespace, key, value)
store.put(namespace, "job", {"text": "Works at Zensar Technologies"})
store.put(namespace, "unit_pref", {"text": "Prefers weather in Celsius"})

# get a specific item back
item = store.get(namespace, "job")
print("Direct get:", item.value)

# search returns everything in the namespace (or filtered by query -- see Approach 3)
print()
print("All facts for user_sharad:")
for result in store.search(namespace):
    print(" -", result.value["text"])


Direct get: {'text': 'Works at Zensar Technologies'}

All facts for user_sharad:
 - Works at Zensar Technologies
 - Prefers weather in Celsius


### ⚠️ Notice
This `store` knows nothing about `thread_id` at all — it's a completely separate structure from the `checkpointer`. That separation is the whole point: **conversation history** and **durable facts about a person** are different kinds of memory with different lifetimes.

---
## Approach 2 — Memory Tools: Let the Agent Manage Its Own Memory

Instead of us manually deciding what to save, we give the agent two tools:
- `save_memory` — call this when the user shares something worth remembering
- `recall_memories` — call this before answering "what do you know about me?"-style questions

Inside a tool, `get_store()` and `get_config()` (from `langgraph.config`) give access to the `store` and the current invocation's `config` — that's how a tool finds out *which user* it's saving/recalling for.

In [6]:
from langgraph.config import get_store, get_config

@tool
def save_memory(fact: str) -> str:
    """Save an important fact about the user for future conversations
    (e.g. their name, workplace, or a stated preference)."""
    store = get_store()
    user_id = get_config()["configurable"]["user_id"]
    store.put(("memories", user_id), fact[:40], {"text": fact})
    return f"Saved: {fact}"

@tool
def recall_memories(query: str) -> str:
    """Search saved facts about the user relevant to the query.
    ALWAYS call this before claiming you don't know something about the user."""
    store = get_store()
    user_id = get_config()["configurable"]["user_id"]
    results = store.search(("memories", user_id), query=query, limit=5)
    if not results:
        return "No relevant memories found."
    return "\n".join(f"- {r.value['text']}" for r in results)

print("Memory tools ready: save_memory, recall_memories")


Memory tools ready: save_memory, recall_memories


In [7]:
long_term_store = InMemoryStore()

agent_with_ltm = create_agent(
    model=llm,
    tools=[get_weather, get_stock_price, save_memory, recall_memories],
    system_prompt=(
        "You are a helpful assistant with long-term memory. "
        "When the user shares a personal fact, call save_memory, then briefly confirm. "
        "When asked what you know about the user, ALWAYS call recall_memories first. "
        "Your final answer MUST be based only on the tool output returned -- restate those facts plainly. "
        "Never say you don't know something if the tool result contains it."
    ),
    checkpointer=MemorySaver(),   # short-term: conversation flow within a thread
    store=long_term_store,        # long-term: facts, shared across all threads for a user
)

print("Agent with both checkpointer (short-term) and store (long-term) ready.")


Agent with both checkpointer (short-term) and store (long-term) ready.


In [8]:
# NOTE: config now carries BOTH a thread_id (conversation) AND a user_id (identity)

# Session 1 -- thread "chat_2026_07_01", user "sharad_2001"
cfg_s1 = {"configurable": {"thread_id": "chat_2026_07_01", "user_id": "sharad_2001"}}
r1 = agent_with_ltm.invoke(
    {"messages": [("user", "Hi, I'm Sharad. I work at Zensar Technologies and I prefer Celsius for weather.")]},
    config=cfg_s1,
)
print("Session 1 -- Agent:", r1["messages"][-1].content)

print()

# Session 2 -- a BRAND NEW thread ("chat_2026_07_08"), SAME user_id
cfg_s2 = {"configurable": {"thread_id": "chat_2026_07_08", "user_id": "sharad_2001"}}
r2 = agent_with_ltm.invoke(
    {"messages": [("user", "What do you know about me?")]},
    config=cfg_s2,
)
print("Session 2 (new thread, same user) -- Agent:", r2["messages"][-1].content)
print()
print("The agent now recognizes the returning user across completely separate threads.")


Session 1 -- Agent: Nice to meet you, Sharad! I’ve noted that you work at Zensar Technologies and that you prefer temperatures in Celsius. Let me know how I can help you today.

Session 2 (new thread, same user) -- Agent: Here’s what I have stored about you:

- **Name:** Sharad  
- **Workplace:** Zensar Technologies  
- **Weather preference:** You prefer temperatures in Celsius.

The agent now recognizes the returning user across completely separate threads.


In [9]:
# Isolation check -- a DIFFERENT user should get nothing back

cfg_priya = {"configurable": {"thread_id": "chat_priya_1", "user_id": "priya_2002"}}
r3 = agent_with_ltm.invoke(
    {"messages": [("user", "What do you know about me?")]},
    config=cfg_priya,
)
print("Different user (priya_2002) -- Agent:", r3["messages"][-1].content)
print()
print("Long-term memory is isolated per user_id, just like threads were isolated per thread_id.")


Different user (priya_2002) -- Agent: I don’t have any personal information about you stored in my memory yet. If you’d like me to remember something—like your name, interests, or any other details—just let me know and I’ll save it for future conversations.

Long-term memory is isolated per user_id, just like threads were isolated per thread_id.


---
## Approach 3 — Semantic Recall: Finding Memories by Meaning

`store.search()` can do more than list everything in a namespace — if we give the store an **embedding model**, it can rank results by *semantic similarity* to a query. This matters once a user has dozens of saved facts and the agent needs the 2-3 that are actually relevant.

We'll use `granite-embedding` (already pulled locally via Ollama) as the embedding model.

In [10]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="granite-embedding")

semantic_store = InMemoryStore(index={"embed": embeddings, "dims": None, "fields": ["text"]})

ns = ("memories", "user_sharad")
semantic_store.put(ns, "fact1", {"text": "User prefers weather updates in Celsius"})
semantic_store.put(ns, "fact2", {"text": "User works at Zensar Technologies in Pune"})
semantic_store.put(ns, "fact3", {"text": "User likes cricket and follows the Indian team"})

# A query that doesn't share exact words with any stored fact
results = semantic_store.search(ns, query="what temperature unit does the user like?", limit=2)

print("Semantic search results (ranked by relevance):")
for r in results:
    print(f"  score={r.score:.3f}  ->  {r.value['text']}")


Semantic search results (ranked by relevance):
  score=0.762  ->  User prefers weather updates in Celsius
  score=0.636  ->  User likes cricket and follows the Indian team


### 🔍 What just happened?
The query never said "Celsius" or "weather" — it asked about a "temperature unit." The embedding model matched it to the *meaning* of the stored fact, not its exact wording, and ranked it above the unrelated cricket fact. This is what lets an agent scale to hundreds of memories without stuffing all of them into every prompt.

---
## Approach 4 — `SqliteStore`: Long-Term Memory That Survives a Restart

`InMemoryStore` lives in RAM — restart the app and every fact is gone. For production, facts about your users need to be on disk (or a real database), same as we did for conversation history in Lesson A.

In [11]:
from langgraph.store.sqlite import SqliteStore

db_path = "long_term_memory.db"

with SqliteStore.from_conn_string(db_path) as persistent_store:
    persistent_store.setup()  # creates tables on first run

    persistent_store.put(("memories", "sharad_2001"), "job", {"text": "Works at Zensar Technologies"})
    persistent_store.put(("memories", "sharad_2001"), "unit_pref", {"text": "Prefers Celsius"})

    print("Saved to disk. Simulating an app restart now...")




Saved to disk. Simulating an app restart now...


In [12]:
from langgraph.store.sqlite import SqliteStore

db_path = "long_term_memory.db"
# Re-open the same file in a fresh `with` block -- like a new process starting up
with SqliteStore.from_conn_string(db_path) as reopened_store:
    reopened_store.setup()
    facts = list(reopened_store.search(("memories", "sharad_2001")))
    print()
    print("Facts recovered after 'restart':")
    for f in facts:
        print(" -", f.value["text"])


Facts recovered after 'restart':
 - Works at Zensar Technologies
 - Prefers Celsius


In [13]:
import os
print(f"Database file: {db_path}")
print(f"Size on disk:  {os.path.getsize(db_path)} bytes")
print()
print("This file survives kernel restarts, server reboots, and deployments --")
print("just like conversation_memory.db did for short-term memory in Lesson A.")


Database file: long_term_memory.db
Size on disk:  24576 bytes

This file survives kernel restarts, server reboots, and deployments --
just like conversation_memory.db did for short-term memory in Lesson A.


---
## 📌 Summary

### What we learned

**The gap:** Lesson A's checkpointer solves memory *within* a thread. A new thread — a returning customer's next visit — still starts blank.

**The fix:** A second store, namespaced by `user_id` instead of `thread_id`, holding distilled facts instead of full transcripts.

| Approach | Code change | Use when |
|---|---|---|
| `InMemoryStore` (manual) | `store.put(ns, key, value)` / `store.search(ns)` | Prototyping, understanding the primitive |
| Memory tools (`save_memory`/`recall_memories`) | Give the agent `get_store()` + `get_config()` inside tools | Letting the agent decide what's worth remembering |
| Semantic recall | `InMemoryStore(index={"embed": ..., "fields": [...]})` | Many stored facts; need the *relevant* ones, not all of them |
| `SqliteStore` | `SqliteStore.from_conn_string(path)` | Production — facts must survive restarts |

### Key concept: `checkpointer` + `store` together
- `checkpointer` → **short-term**, scoped to `thread_id` → *what was said*
- `store` → **long-term**, scoped to `user_id` → *what's true about this person*
- A production agent almost always uses **both** at once, as `agent_with_ltm` did above.

### What's next — Lesson C: Explicit LangGraph
So far `create_agent` has been building the graph for us under the hood. In **Lesson C** we'll construct it ourselves — nodes, edges, and conditional routing — to get full control over multi-step reasoning, branching logic, and where human approval steps fit in.

---
## 🎁 Bonus — Returning Customer, End-to-End

A realistic support scenario: a customer chats today, then comes back next week in a brand-new session. The agent should recognize them immediately.

In [14]:
bonus_store = InMemoryStore()

support_agent = create_agent(
    model=llm,
    tools=[get_weather, get_stock_price, save_memory, recall_memories],
    system_prompt=(
        "You are a friendly Zensar support assistant with long-term memory. "
        "Save important facts about returning customers with save_memory. "
        "ALWAYS call recall_memories before answering questions about what you know of the user, "
        "and base your answer only on what the tool returns."
    ),
    checkpointer=MemorySaver(),
    store=bonus_store,
)

user_id = "rahul_mumbai_884"

print("=" * 60)
print("WEEK 1 -- First contact")
week1 = {"configurable": {"thread_id": "ticket_991", "user_id": user_id}}
r = support_agent.invoke(
    {"messages": [("user", "Hi, I'm Rahul from the Mumbai office. I'm tracking Zensar's stock and I prefer email updates.")]},
    config=week1,
)
print("Agent:", r["messages"][-1].content)

print()
print("=" * 60)
print("WEEK 2 -- New ticket, new thread_id, same customer")
week2 = {"configurable": {"thread_id": "ticket_1042", "user_id": user_id}}
r = support_agent.invoke(
    {"messages": [("user", "Hi again, quick question -- what's the Zensar stock price today?")]},
    config=week2,
)
print("Agent:", r["messages"][-1].content)

r = support_agent.invoke(
    {"messages": [("user", "By the way, do you remember anything about me from before?")]},
    config=week2,
)
print("Agent:", r["messages"][-1].content)


WEEK 1 -- First contact
Agent: 

WEEK 2 -- New ticket, new thread_id, same customer
Agent: The current price for Zensar Technologies (ticker **ZENSAR**) is **₹ 464.00**. Let me know if you need anything else—historical data, market news, or help with Zensar services!
Agent: Hey Rahul! 👋 I see you’re based in the Mumbai office. Let me know how I can help you today—whether it’s another quick check on Zensar stock, a question about our services, or anything else you need.
